In [1]:
import os
import re
import sys
import copy
import subprocess
import yaml

import ipywidgets as widgets
import pandas as pd
import numpy as np
from IPython.display import display

sys.path.append('./src')
from arc import Arc, ArcList
from node import Node, NodeList

allowed_pattern = re.compile(r'^[0-9.]*$')

nodes_list = NodeList()
nodes_list.add_node(Node(name="A", value=""))
nodes_list.add_node(Node(name="G", value="0"))

arcs_list = ArcList()
arcs_list.add_arc(Arc(node_1='A', node_2='G', value_1="", value_2=""))

tiebreaker_value = "alphabetical order"

def on_tiebreaker_value_change(change):
    global tiebreaker_value
    tiebreaker_value = change['new']
    update_nodes_ui()

tiebreaker_radio = widgets.RadioButtons(
    options=["alphabetical order", "order of nodes"],
    value="alphabetical order",
    style={'description_width': 'initial'}
)
tiebreaker_radio.observe(on_tiebreaker_value_change, names='value')

def generate_yaml(exam_name, exam_date, nodes, nodes_ord, arcs, source, destination, graph_ratio, level_ratio, label_offset, heuristic_offset, tiebreaker):
    str_ord = "" if tiebreaker == "alphabetical order" else nodes_ord
    data = f"""
exam: {exam_name}
date: {exam_date}
exercises:
  search:
    source: {source}
    destination: {destination}
    tiebreaker: {tiebreaker}
    nodes:
{nodes}    arcs:
{arcs}{str_ord}    graph_ratio: {graph_ratio}
    level_ratio: {level_ratio}
    label_offset: {label_offset}
    heuristic_offset: {heuristic_offset}
"""
    
    try:
        with open('../sources/file.yaml', 'w') as file:
            file.write(data)
        print("YAML file created successfully!")

        subprocess.run(['python', 'main.py'], cwd="../scripts")
        print("Exam solution generated successfully")
    except Exception as e:
        print(f"Error creating the file: {e}")

def on_arc_text_change(change):
    global arcs_list
    if change['type'] == 'change' and change['name'] == 'value':
        if not allowed_pattern.match(change['new']):
            change['owner'].value = change['old']
            return
        id = change['owner'].custom_id
        pos = change['owner'].custom_pos
        node_1, node_2 = id.split("_")
        for arc in arcs_list.arcs:
            if arc.has_equal_nodes_params(node_1, node_2) or arc.has_equal_nodes_params(node_2, node_1):
                if pos == 0:
                    arc.value_1 = change['new']
                elif pos == 1:
                    arc.value_2 = change['new']
                break

def on_node_value_change(change):
    global nodes_list
    if change['type'] == 'change' and change['name'] == 'value':
        if not allowed_pattern.match(change['new']):
            change['owner'].value = change['old']
            return
        node_name = change['owner'].custom_id
        new_value = change['new']
        for node in nodes_list:
            if node.name == node_name:
                node.value = new_value
                break

def update_nodes_ui():
    node_widgets = []
    for node in nodes_list:
        node_label = widgets.HTML(f"<b>{node.name}:</b>")
        node_input_field = widgets.Text(value=node.value, layout=widgets.Layout(width='60px'))
        node_input_field.custom_id = node.name
        node_input_field.observe(on_node_value_change, names='value')

        if tiebreaker_value != "alphabetical order":
            dropdown = widgets.Dropdown(
                options=['-'] + [str(i) for i in range(1, len(nodes_list) + 1)],
                value=node.ord if node.ord else '-',
                layout=widgets.Layout(width='60px')
            )

            def make_handler(node):
                def handler(change):
                    node.ord = change['new'] if change['new'] != '-' else ''
                return handler

            dropdown.observe(make_handler(node), names='value')
            node_box = widgets.HBox([node_input_field, dropdown])
        else:
            node_box = widgets.HBox([node_input_field])

        node_widgets.append(
            widgets.VBox([
                node_label,
                node_box
            ], layout=widgets.Layout(margin='0 0 4px', width='100%'))
        )

    nodes_ui_container.children = node_widgets

def update_arcs_ui():
    global arcs_list
    old_values = {(arc.node_1, arc.node_2): (arc.value_1, arc.value_2) for arc in arcs_list.arcs}
    arcs_list_temp = ArcList()
    for i, node_a in enumerate(nodes_list.get_list_names()):
        for node_b in nodes_list.get_list_names()[i+1:]:
            arc = Arc(node_1=node_a, node_2=node_b, value_1="", value_2="")
            if (node_a, node_b) in old_values:
                arc.value_1, arc.value_2 = old_values[(node_a, node_b)]
            elif (node_b, node_a) in old_values:
                arc.value_2, arc.value_1 = old_values[(node_b, node_a)]
            arcs_list_temp.add_arc(arc)
    arcs_list = arcs_list_temp
    arc_widgets = []
    for arc in arcs_list.arcs:
        cell_1 = widgets.HTML(f"<b>{arc.node_1} → {arc.node_2}:</b>")
        cell_2 = widgets.HTML(f"<b>{arc.node_2} → {arc.node_1}:</b>")
        value_1_input = widgets.Text(value=arc.value_1, layout=widgets.Layout(width='60px'))
        value_1_input.custom_id = f"{arc.node_1}_{arc.node_2}"
        value_1_input.custom_pos = 0
        value_2_input = widgets.Text(value=arc.value_2, layout=widgets.Layout(width='60px'))
        value_2_input.custom_id = f"{arc.node_2}_{arc.node_1}"
        value_2_input.custom_pos = 1
        value_1_input.observe(on_arc_text_change, names='value')
        value_2_input.observe(on_arc_text_change, names='value')
        arc_widgets.append(
            widgets.HBox([
                widgets.VBox([cell_1, value_1_input]),
                widgets.VBox([cell_2, value_2_input])
            ], layout=widgets.Layout(justify_content='space-between', width='100%'))
        )
    arcs_ui_container.children = arc_widgets

def update_nodes(change=None):
    global nodes_list
    nodes = re.split(r'[ ,;_]+', node_input.value.strip())
    nodes = [node.strip() for node in nodes if node.strip()]
    unique_nodes = list(dict.fromkeys(nodes))
    unique_nodes_list = NodeList()
    for node in unique_nodes:
        value = ""
        if node in nodes_list.get_list_names():
            value = nodes_list.get_value(node)
        unique_nodes_list.add_node(Node(name=node, value=value))
    nodes_list = unique_nodes_list

    source_value = source_input.value if source_input.value in nodes_list.get_list_names() else None
    destination_value = destination_input.value if destination_input.value in nodes_list.get_list_names() else None
    source_input.options = nodes_list.get_list_names()
    destination_input.options = nodes_list.get_list_names()
    source_input.value = source_value or (nodes_list.get_list_names()[0] if nodes_list.get_list_names() else None)
    destination_input.value = destination_value or (nodes_list.get_list_names()[-1] if nodes_list.get_list_names() else None)
    current_nodes.clear_output(wait=True)

    with current_nodes:
        if unique_nodes:
            node_widgets = [
                widgets.HTML(f'<div style="padding: 3px 10px; margin: 3px; border: 1px solid #ccc; '
                             f'border-radius: 6px; background-color: #e0e0e0; font-size: 12px;">{node}</div>')
                for node in unique_nodes
            ]
            display(widgets.HBox(node_widgets))
        else:
            display(widgets.HTML("<i>No nodes added.</i>"))

    update_nodes_ui()
    update_arcs_ui()

exam_name_input = widgets.Text(value='Artificial Intelligence Exam', description='Exam:')
exam_date_input = widgets.DatePicker(value=pd.to_datetime('today').date(), description='Date:', tooltip='')
node_input = widgets.Text(value=" ".join(nodes_list.get_list_names()), description='Nodes:')
node_input.observe(update_nodes, names='value')
current_nodes = widgets.Output()

arcs_ui_container = widgets.VBox()
nodes_ui_container = widgets.VBox()

source_input = widgets.Dropdown(options=nodes_list.get_list_names(), value='A', description='Source:')
destination_input = widgets.Dropdown(options=nodes_list.get_list_names(), value='G', description='Destination:')
graph_ratio_input = widgets.FloatSlider(value=1, min=0, max=5, step=0.1, description='Graph Ratio:', tooltip='the width of the input text figure')
level_ratio_input = widgets.FloatSlider(value=7, min=0, max=10, step=0.1, description='Level Ratio:', tooltip='the width of the output solution figure')
label_offset_input = widgets.FloatSlider(value=0.13, min=0, max=1, step=0.01, description='Label Offset:', tooltip='the label offset of the output solution figure')
heuristic_offset_input = widgets.FloatSlider(value=0.1, min=0, max=1, step=0.01, description='Heuristic Offset:', tooltip='the heuristic offset of the output solution figure', style={'description_width': 'initial'})

generate_button = widgets.Button(description="Generate Exam Solution", button_style='success', layout=widgets.Layout(width='200px'))
generate_button_box = widgets.HBox([generate_button], layout=widgets.Layout(margin='20px 0'))

def on_generate_button_click(b):
    global nodes_list, arcs_list
    list_error = []

    exam_name = exam_name_input.value
    exam_date = str(exam_date_input.value)
    source = source_input.value
    destination = destination_input.value
    graph_ratio = graph_ratio_input.value
    level_ratio = level_ratio_input.value
    label_offset = label_offset_input.value
    heuristic_offset = heuristic_offset_input.value
    tiebreaker = tiebreaker_radio.value

    if not exam_name:
        list_error.append("Exam name is empty.")
    if not exam_date:
        list_error.append("Exam date is empty.")
    if not nodes_list.get_list_names():
        list_error.append("Node list is empty.")
    elif nodes_list.get_nodes_values_str() is None:
        list_error.append("Nodes must have values.")
    if tiebreaker == "order of nodes" and not nodes_list.validate():
        list_error.append("Each node must have a unique 'ord' for order of nodes.")
    if not arcs_list.arcs:
        list_error.append("Arc list is empty.")
    elif arcs_list.get_arcs_values_str() is None:
        list_error.append("No valid arcs specified.")
    if not source:
        list_error.append("No source selected.")
    if not destination:
        list_error.append("No destination selected.")

    if list_error:
        for error in list_error:
            print(f"❌ {error}")
        return

    generate_yaml(
        exam_name,
        exam_date,
        nodes_list.get_nodes_values_str(),
        nodes_list.get_nodes_ords_str(),
        arcs_list.get_arcs_values_str(),
        source,
        destination,
        graph_ratio,
        level_ratio,
        label_offset,
        heuristic_offset,
        tiebreaker
    )

generate_button.on_click(on_generate_button_click)

form_section = widgets.VBox([
    widgets.HTML("<h3 style='margin-bottom: 0px;margin-top: 20px;'>Exam Information</h3>"),
    exam_name_input,
    exam_date_input,
    widgets.HTML("<h3 style='margin-bottom: 0px;margin-top: 20px;'>Nodes</h3>"),
    node_input,
    current_nodes,
    widgets.HTML("<h3 style='margin-bottom: 0px;margin-top: 20px;'>Goals</h3>"),
    source_input,
    destination_input,
    widgets.HTML("<h3 style='margin-bottom: 0px;margin-top: 20px;'>Parameters</h3>"),
    graph_ratio_input,
    level_ratio_input,
    label_offset_input,
    heuristic_offset_input,
    widgets.HTML(
        "<h3 style='margin-bottom: 0px;margin-top: 20px;'>"
        "Expansion mode in case of equal f(n) values "
        "<span title=\"Choose how to break ties when nodes have the same f(n) value.&#10;"
        "'Alphabetical order' expands nodes in alphabetical order.&#10;"
        "'Order of nodes' expands nodes based on the assigned custom order.\">🛈</span></h3>"
    ),
    tiebreaker_radio,
    generate_button_box
], layout=widgets.Layout(width='50%', padding='10px'))

nodes_section = widgets.VBox([
    widgets.HTML("<h3 style='margin-bottom: 0px;margin-top: 20px;'>Heuristic <span title=\"This section allows you to set heuristic values for each node.&#10;If 'Order of nodes' mode is selected, each node must be assigned an expansion order position.\">🛈</span></h3>"),
    nodes_ui_container
], layout=widgets.Layout(width='48%', padding='10px'))

arcs_section = widgets.VBox([
    widgets.HTML("<h3 style='margin-bottom: 0px;margin-top: 20px;'>Arcs <span title='This section allows you to define the arcs between nodes with their respective values.&#10;Leave empty if the arc is not expected.'>🛈</span></h3>"),
    arcs_ui_container
], layout=widgets.Layout(width='48%', padding='10px'))

nodes_arcs_VBox_section = widgets.VBox([
    nodes_section,
    arcs_section
], layout=widgets.Layout(width='48%', padding='10px'))

dashboard = widgets.HBox([form_section, nodes_arcs_VBox_section])
display(dashboard)

update_nodes()
